# **Reproducción de audios**

## **Librerías y módulos necesarios**

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import Audio, display
import os

## **Datos**

In [2]:
audio_dir = '../data/train_audio'

csv_filename = '../data/observaciones_top20_especies.csv'

try:
    df = pd.read_csv(csv_filename)
    df
    
except FileNotFoundError:
    print(f"\nERROR: El archivo '{csv_filename}' no se encuentra en el directorio actual.")

In [3]:
if 'df' in locals() and not df.empty:
    try:
        available_files = set(os.listdir(audio_dir))
        print(f"Archivos encontrados en '{audio_dir}': {len(available_files)}")
    except FileNotFoundError:
        print(f"ERROR: La carpeta '{audio_dir}' no se encontró. Asegúrate de que el nombre sea correcto.")
        available_files = set()
else:
    print("ERROR: El DataFrame 'df' está vacío o no se cargó correctamente en la Celda 1.")
    available_files = set()

Archivos encontrados en '../data/train_audio': 146


In [4]:
FILENAME_COLUMN = 'primary_label'

if FILENAME_COLUMN in df.columns:
    
    df_filtered = df[df[FILENAME_COLUMN].isin(available_files)].copy()
    
    if df_filtered.empty:
        print("Advertencia: El DataFrame filtrado está vacío. No hay coincidencias entre el CSV y la carpeta de audios.")

else:
    print(f"\n ERROR: La columna '{FILENAME_COLUMN}' no se encontró en el CSV.")
    
    df_filtered = pd.DataFrame()

In [5]:
if not df_filtered.empty:

    df_filtered['combined_label'] = (
        df_filtered['primary_label'].astype(str) + 
        ' (' + df_filtered['scientific_name'].astype(str) + ')'
    )
    
    species_options = sorted(df_filtered['combined_label'].unique().tolist())
    
    if species_options:
        initial_species = species_options[0]
        initial_audios = sorted(df_filtered[
            df_filtered['combined_label'] == initial_species
        ][FILENAME_COLUMN].tolist())
    else:
        initial_species = "No data"
        initial_audios = ["No hay audios disponibles"]
    
else:
  
    species_options = ['No data']
    initial_species = "No data"
    initial_audios = ["No hay audios disponibles"]

## **Funciones**

In [6]:
species_dropdown = widgets.Dropdown(
    options=species_options,
    value=initial_species if species_options else None,
    description='Seleccionar Especie:',
    disabled=not bool(species_options),
    style={'description_width': 'initial'} # Ajusta el ancho para que quepa la descripción
)


audio_dropdown = widgets.Dropdown(
    options=initial_audios,
    value=initial_audios[0] if initial_audios and initial_audios[0] != "No hay audios disponibles" else None,
    description='Seleccionar Audio:',
    disabled=not bool(initial_audios) or initial_audios[0] == "No hay audios disponibles",
    style={'description_width': 'initial'}
)

audio_output = widgets.Output()

In [7]:
def update_audio_options(change):
    
    selected_species_label = change.new
    
    with audio_output:
        audio_output.clear_output() 
        
    if selected_species_label and selected_species_label != 'No data':
        new_audios = sorted(df_filtered[
            df_filtered['combined_label'] == selected_species_label
        ][FILENAME_COLUMN].tolist())
        
        audio_dropdown.options = new_audios
        audio_dropdown.value = new_audios[0] if new_audios else None
        audio_dropdown.disabled = not bool(new_audios)
    else:
        
        audio_dropdown.options = ["No hay audios disponibles"]
        audio_dropdown.value = None
        audio_dropdown.disabled = True

    if audio_dropdown.value:
         
        display_audio_player(widgets.widget.trait_types.Result({"new": audio_dropdown.value}))
    else:
        with audio_output:
            audio_output.clear_output()


def display_audio_player(change):
    
    selected_filename = change.new
    
    with audio_output:
        audio_output.clear_output() 
        
        if selected_filename and selected_filename != "No hay audios disponibles":
            audio_path = os.path.join(audio_dir, selected_filename)
            print(f"Reproduciendo: {selected_filename}")
            
            display(Audio(audio_path))
        else:
             print("Selecciona una especie y un audio para reproducir.")

species_dropdown.observe(update_audio_options, names='value')

audio_dropdown.observe(display_audio_player, names='value')

In [ ]:
print("INTERFAZ PARA REPRODUCCIÓN DE AUDIOS ")
print("1. Selecciona una especie (código - nombre científico).")
print("2. Selecciona un audio para reproducirlo.")
print("-" * 50)

interface_layout = widgets.VBox([
    species_dropdown,
    audio_dropdown
])

display(interface_layout, audio_output)

if audio_dropdown.value:
    display_audio_player(widgets.widget.trait_types.Result({"new": audio_dropdown.value}))